# Case-Only EDA (Legacy)

**Source script:** `run_eda_case_only.py`

Notebook mirror for submission traceability.

In [ ]:
import json
import math
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path(__file__).resolve().parent
ENRICHED_PATH = ROOT / "outputs" / "imputed_dataset_enriched.csv"
PRE_PATH = ROOT / "outputs" / "imputed_dataset.csv"
ENRICH_REPORT_PATH = ROOT / "outputs" / "enrichment_report.json"
GEOCODE_PATH = ROOT / "geocoded_locations.csv"
UV_CACHE_DIR = ROOT / "cache_uv" / "temis_msr2"

OUT_DIR = ROOT / "outputs" / "eda"
PLOTS_DIR = OUT_DIR / "plots"
TABLES_DIR = OUT_DIR / "tables"
OUT_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid")


def sanitize_filename(name: str) -> str:
    name = re.sub(r"[^A-Za-z0-9._-]+", "_", name)
    return name.strip("_")


def save_plot(fig, filename: str):
    path = PLOTS_DIR / filename
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    plt.close(fig)


def is_numeric_series(s: pd.Series) -> bool:
    return pd.api.types.is_numeric_dtype(s)


def series_diff_count(a: pd.Series, b: pd.Series) -> int:
    if is_numeric_series(a) and is_numeric_series(b):
        a_vals = a.to_numpy(dtype=float, copy=False)
        b_vals = b.to_numpy(dtype=float, copy=False)
        neq = ~np.isclose(a_vals, b_vals, equal_nan=True)
        return int(np.nansum(neq))
    a_f = a.fillna("__NA__")
    b_f = b.fillna("__NA__")
    return int((a_f != b_f).sum())



_df = pd.read_csv(ENRICHED_PATH)
_pre = pd.read_csv(PRE_PATH)


if ENRICH_REPORT_PATH.exists():
    with open(ENRICH_REPORT_PATH) as f:
        enrich_report = json.load(f)
else:
    enrich_report = {}


_df["presentation_date"] = pd.to_datetime(_df["presentation_date"], errors="coerce")


cols = list(_df.columns)
uv_cols = [c for c in cols if c.startswith("uv_")]
moon_cols = [c for c in cols if c.startswith("moon_") or c in {"is_full_moon", "is_new_moon"}]

weather_keywords = [
    "temperature",
    "shortwave",
    "precip",
    "rain",
    "snow",
    "humidity",
    "wind",
    "surface_pressure",
    "pressure",
    "cloud",
    "evap",
    "radiation",
    "sunshine",
    "dewpoint",
    "vapour",
    "vapor",
    "et0",
]
weather_cols = [c for c in cols if any(k in c.lower() for k in weather_keywords)]

meta_candidates = [
    "ID",
    "zip code",
    "zip5",
    "presentation_date",
    "date of presentation",
    "date of birth",
    "weather_job_id",
    "weather_missing_reason",
    "town/village",
    "state",
    "urban or rural region",
    "altitude (MASL)",
    "weather station",
    "clostest Weather station ID",
    "name",
]
meta_cols = [c for c in meta_candidates if c in cols]

clinical_cols = [
    c
    for c in cols
    if c not in set(uv_cols + moon_cols + weather_cols + meta_cols)
]


integrity = {
    "row_count": len(_df),
    "column_count": len(_df.columns),
    "unique_IDs": _df["ID"].nunique(dropna=True) if "ID" in _df.columns else np.nan,
    "ID_unique": _df["ID"].is_unique if "ID" in _df.columns else np.nan,
    "ID_plus_presentation_date_unique": None,
    "presentation_date_missing": int(_df["presentation_date"].isna().sum()),
}
if {"ID", "presentation_date"}.issubset(_df.columns):
    integrity["ID_plus_presentation_date_unique"] = (
        _df[["ID", "presentation_date"]].duplicated().sum() == 0
    )


pre_cols = list(_pre.columns)
new_cols = [c for c in cols if c not in pre_cols]
dropped_cols = [c for c in pre_cols if c not in cols]


common_cols = [c for c in pre_cols if c in cols]
non_uv_moon_common = [
    c for c in common_cols if c not in set(uv_cols + moon_cols)
]
value_diff_rows = []
for c in non_uv_moon_common:
    diff_n = series_diff_count(_pre[c], _df[c])
    if diff_n > 0:
        value_diff_rows.append({"column": c, "diff_count": diff_n})


missing_n = _df.isna().sum()
missing_pct = (missing_n / len(_df)) * 100.0
missing_table = pd.DataFrame({
    "variable": missing_n.index,
    "missing_n": missing_n.values,
    "missing_pct": missing_pct.values,
}).sort_values("missing_pct", ascending=False)
missing_table.to_csv(TABLES_DIR / "missingness_per_column.csv", index=False)


def block_missing_summary(block_cols):
    if not block_cols:
        return pd.Series({
            "columns": 0,
            "missing_mean_pct": np.nan,
            "missing_median_pct": np.nan,
            "missing_max_pct": np.nan,
        })
    block_missing = _df[block_cols].isna().mean() * 100.0
    return pd.Series({
        "columns": len(block_cols),
        "missing_mean_pct": block_missing.mean(),
        "missing_median_pct": block_missing.median(),
        "missing_max_pct": block_missing.max(),
    })

block_missing = pd.DataFrame({
    "uv": block_missing_summary(uv_cols),
    "moon": block_missing_summary(moon_cols),
    "weather": block_missing_summary(weather_cols),
    "clinical": block_missing_summary(clinical_cols),
}).T
block_missing = block_missing.reset_index().rename(columns={"index": "block"})
block_missing.to_csv(TABLES_DIR / "missingness_by_block.csv", index=False)


fig, ax = plt.subplots(figsize=(10, 6))
missing_table.head(30).plot(
    x="variable",
    y="missing_pct",
    kind="bar",
    ax=ax,
    color="#4C78A8",
)
ax.set_title("Top 30 Missingness Percent")
ax.set_ylabel("Missing %")
ax.set_xlabel("")
ax.tick_params(axis="x", labelrotation=90)
save_plot(fig, "missingness_top30_bar.png")


sorted_cols = missing_table["variable"].tolist()
miss_matrix = _df[sorted_cols].isna().astype(int)
fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(miss_matrix, cbar=False, ax=ax)
ax.set_title("Missingness Heatmap (Columns sorted by missingness)")
ax.set_xlabel("Variables")
ax.set_ylabel("Rows")
ax.set_xticks([])
ax.set_yticks([])
save_plot(fig, "missingness_heatmap.png")



if uv_cols:
    n = len(uv_cols)
    cols_n = 3
    rows_n = math.ceil(n / cols_n)
    fig, axes = plt.subplots(rows_n, cols_n, figsize=(12, 3 * rows_n))
    axes = np.array(axes).reshape(-1)
    for i, c in enumerate(uv_cols):
        sns.histplot(_df[c], kde=True, ax=axes[i], color="#59A14F")
        axes[i].set_title(c)
    for j in range(i + 1, len(axes)):
        axes[j].axis("off")
    save_plot(fig, "uv_histograms.png")


if uv_cols:
    uv_melt = _df[uv_cols].melt(var_name="uv_feature", value_name="value")
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.boxplot(data=uv_melt, x="uv_feature", y="value", ax=ax, color="#F28E2B")
    ax.set_title("UV Feature Distributions")
    ax.tick_params(axis="x", labelrotation=45)
    save_plot(fig, "uv_feature_boxplot.png")


moon_numeric = [c for c in moon_cols if _df[c].dtype != "object"]
for c in moon_numeric:
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.histplot(_df[c], kde=True, ax=ax, color="#B07AA1")
    ax.set_title(f"Moon Distribution: {c}")
    save_plot(fig, f"moon_{sanitize_filename(c)}_hist.png")

if "moon_phase_category" in _df.columns:
    fig, ax = plt.subplots(figsize=(8, 4))
    _df["moon_phase_category"].value_counts().sort_index().plot(
        kind="bar", ax=ax, color="#9C755F"
    )
    ax.set_title("Moon Phase Category Counts")
    ax.set_xlabel("")
    save_plot(fig, "moon_phase_category_bar.png")


weather_show = [
    c
    for c in [
        "temperature_2m_mean__day1",
        "temperature_2m_max__day1",
        "shortwave_radiation_sum__day1",
    ]
    if c in _df.columns
]
if weather_show:
    fig, axes = plt.subplots(1, len(weather_show), figsize=(5 * len(weather_show), 4))
    if len(weather_show) == 1:
        axes = [axes]
    for ax, c in zip(axes, weather_show):
        sns.histplot(_df[c], kde=True, ax=ax, color="#4E79A7")
        ax.set_title(c)
    save_plot(fig, "weather_key_histograms.png")


_df["month"] = _df["presentation_date"].dt.month
_df["month_name"] = _df["presentation_date"].dt.month_name()

season_map = {
    12: "DJF", 1: "DJF", 2: "DJF",
    3: "MAM", 4: "MAM", 5: "MAM",
    6: "JJA", 7: "JJA", 8: "JJA",
    9: "SON", 10: "SON", 11: "SON",
}
_df["season"] = _df["month"].map(season_map)


if "uv_index_max__day1" in _df.columns:
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.boxplot(data=_df, x="month", y="uv_index_max__day1", ax=ax, color="#59A14F")
    ax.set_title("UV Index (day1) by Month")
    save_plot(fig, "uv_day1_by_month_boxplot.png")


if "shortwave_radiation_sum__day1" in _df.columns:
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.boxplot(data=_df, x="month", y="shortwave_radiation_sum__day1", ax=ax, color="#4E79A7")
    ax.set_title("Shortwave Radiation (day1) by Month")
    save_plot(fig, "shortwave_by_month_boxplot.png")

if "temperature_2m_mean__day1" in _df.columns:
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.boxplot(data=_df, x="month", y="temperature_2m_mean__day1", ax=ax, color="#E15759")
    ax.set_title("Temperature 2m Mean (day1) by Month")
    save_plot(fig, "temp_mean_by_month_boxplot.png")


fig, ax = plt.subplots(figsize=(10, 4))
_df["month"].value_counts().sort_index().plot(kind="bar", ax=ax, color="#76B7B2")
ax.set_title("Case Count by Month")
ax.set_xlabel("Month")
ax.set_ylabel("Count")
save_plot(fig, "case_count_by_month.png")


monthly_vars = [c for c in [
    "uv_index_max__day1",
    "shortwave_radiation_sum__day1",
    "temperature_2m_mean__day1",
] if c in _df.columns]
if monthly_vars:
    monthly = _df.groupby("month")[monthly_vars].mean().sort_index()
    fig, ax = plt.subplots(figsize=(10, 5))
    for c in monthly_vars:
        ax.plot(monthly.index, monthly[c], marker="o", label=c)
    ax.set_title("Monthly Mean Trends")
    ax.set_xlabel("Month")
    ax.legend()
    save_plot(fig, "monthly_mean_trends.png")
    monthly.to_csv(TABLES_DIR / "monthly_means.csv")


numeric_df = _df.select_dtypes(include=["number"]).copy()
exclude_numeric = [c for c in ["ID", "zip5", "zip code"] if c in numeric_df.columns]
numeric_df = numeric_df.drop(columns=exclude_numeric, errors="ignore")


if len(uv_cols) > 1:
    uv_corr = numeric_df[uv_cols].corr()
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(uv_corr, ax=ax, cmap="coolwarm", center=0)
    ax.set_title("UV Correlation")
    save_plot(fig, "uv_correlation_heatmap.png")
else:
    uv_corr = None


if len(weather_cols) > 1:
    weather_corr = numeric_df[weather_cols].corr()
    fig, ax = plt.subplots(figsize=(12, 10))
    sns.heatmap(weather_corr, ax=ax, cmap="coolwarm", center=0)
    ax.set_title("Weather Correlation")
    ax.set_xticks([])
    ax.set_yticks([])
    save_plot(fig, "weather_correlation_heatmap.png")
else:
    weather_corr = None


uv_weather_corr = None
if uv_cols and weather_cols:
    combined = numeric_df[uv_cols + weather_cols]
    corr_all = combined.corr()
    uv_weather_corr = corr_all.loc[uv_cols, weather_cols]
    fig, ax = plt.subplots(figsize=(12, 4))
    sns.heatmap(uv_weather_corr, ax=ax, cmap="coolwarm", center=0)
    ax.set_title("UV vs Weather Correlation")
    save_plot(fig, "uv_weather_correlation_heatmap.png")


high_corr_rows = []
threshold = 0.9

if uv_corr is not None:
    for i, c1 in enumerate(uv_cols):
        for j in range(i + 1, len(uv_cols)):
            c2 = uv_cols[j]
            r = uv_corr.loc[c1, c2]
            if pd.notna(r) and abs(r) >= threshold:
                high_corr_rows.append({"block": "uv", "feature_1": c1, "feature_2": c2, "corr": r})

if weather_corr is not None:
    for i, c1 in enumerate(weather_cols):
        for j in range(i + 1, len(weather_cols)):
            c2 = weather_cols[j]
            r = weather_corr.loc[c1, c2]
            if pd.notna(r) and abs(r) >= threshold:
                high_corr_rows.append({"block": "weather", "feature_1": c1, "feature_2": c2, "corr": r})

if uv_weather_corr is not None:
    for c1 in uv_cols:
        for c2 in weather_cols:
            r = uv_weather_corr.loc[c1, c2]
            if pd.notna(r) and abs(r) >= threshold:
                high_corr_rows.append({"block": "uv_weather", "feature_1": c1, "feature_2": c2, "corr": r})

pd.DataFrame(high_corr_rows).sort_values("corr", key=lambda s: s.abs(), ascending=False).to_csv(
    TABLES_DIR / "high_correlation_pairs.csv", index=False
)


optional_notes = []


if "uv_index_max__day1" in _df.columns:
    uv = _df["uv_index_max__day1"]
    month = _df["month"]
    uv_month_mean = uv.groupby(month).transform("mean")
    uv_month_std = uv.groupby(month).transform("std")
    uv_z = (uv - uv_month_mean) / uv_month_std.replace(0, np.nan)
    _df["uv_day1_month_z"] = uv_z
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.histplot(uv_z.dropna(), kde=True, ax=ax, color="#F28E2B")
    ax.set_title("UV Day1 Monthly Z-Score Distribution")
    save_plot(fig, "uv_day1_monthly_z_hist.png")


if {"uv_index_max__day1", "shortwave_radiation_sum__day1", "month"}.issubset(_df.columns):
    fig, ax = plt.subplots(figsize=(7, 5))
    scatter = ax.scatter(
        _df["shortwave_radiation_sum__day1"],
        _df["uv_index_max__day1"],
        c=_df["month"],
        cmap="viridis",
        alpha=0.7,
    )
    ax.set_title("UV vs Shortwave Radiation (day1)")
    ax.set_xlabel("Shortwave Radiation (day1)")
    ax.set_ylabel("UV Index (day1)")
    fig.colorbar(scatter, ax=ax, label="Month")
    save_plot(fig, "uv_vs_shortwave_scatter.png")


uv_gap_rows = None
if GEOCODE_PATH.exists() and "zip5" in _df.columns:
    geo = pd.read_csv(GEOCODE_PATH)
    if "zip_code" in geo.columns:
        geo = geo[["zip_code", "latitude", "longitude"]]
        geo = geo.groupby("zip_code", as_index=False)[["latitude", "longitude"]].mean()
        df_geo = _df.merge(geo, left_on="zip5", right_on="zip_code", how="left")
        missing_geo = df_geo["latitude"].isna().sum()
        if missing_geo < len(df_geo):
            uv_cache = {}
            raw_missing_flags = []
            for idx, row in df_geo.iterrows():
                date = row.get("presentation_date")
                lat = row.get("latitude")
                lon = row.get("longitude")
                if pd.isna(date) or pd.isna(lat) or pd.isna(lon):
                    raw_missing_flags.append(np.nan)
                    continue
                lat_r = f"{round(float(lat), 3):.3f}"
                lon_r = f"{round(float(lon), 3):.3f}"
                year = int(date.year)
                key = (lat_r, lon_r, year)
                if key not in uv_cache:
                    file_path = UV_CACHE_DIR / f"uv_{lat_r}_{lon_r}_{year}.csv"
                    if not file_path.exists():
                        uv_cache[key] = None
                    else:
                        ser = pd.read_csv(file_path, parse_dates=["date"])
                        ser = ser.set_index("date")["value"]
                        uv_cache[key] = ser
                ser = uv_cache.get(key)
                if ser is None:
                    raw_missing_flags.append(np.nan)
                    continue
                raw_val = ser.get(date.normalize(), np.nan)
                raw_missing_flags.append(pd.isna(raw_val))

            df_geo["uv_raw_missing" ] = raw_missing_flags
            uv_gap_rows = df_geo[(df_geo["uv_raw_missing"] == True) & (df_geo["uv_index_max__day1"].notna())]
            uv_gap_rows[["ID", "presentation_date", "zip5", "uv_index_max__day1"]].to_csv(
                TABLES_DIR / "uv_gapfilled_rows_approx.csv", index=False
            )

            if len(uv_gap_rows) > 0:
                keep = df_geo[df_geo["uv_raw_missing"] != True]
                sens = pd.DataFrame({
                    "subset": ["all_rows", "exclude_gapfilled"],
                    "n": [len(df_geo), len(keep)],
                    "uv_day1_mean": [df_geo["uv_index_max__day1"].mean(), keep["uv_index_max__day1"].mean()],
                    "uv_day1_std": [df_geo["uv_index_max__day1"].std(), keep["uv_index_max__day1"].std()],
                })
                sens.to_csv(TABLES_DIR / "uv_gapfill_sensitivity.csv", index=False)
        else:
            optional_notes.append("Gap-fill sensitivity: no geocode match for zip5.")
    else:
        optional_notes.append("Gap-fill sensitivity: geocode file missing zip_code column.")
else:
    optional_notes.append("Gap-fill sensitivity: geocoded_locations.csv missing or zip5 not present.")


pd.DataFrame([integrity]).to_csv(TABLES_DIR / "integrity_summary.csv", index=False)

pd.DataFrame({
    "new_columns": new_cols,
}).to_csv(TABLES_DIR / "column_diff_new.csv", index=False)

pd.DataFrame({
    "dropped_columns": dropped_cols,
}).to_csv(TABLES_DIR / "column_diff_dropped.csv", index=False)

pd.DataFrame(value_diff_rows, columns=["column", "diff_count"]).to_csv(
    TABLES_DIR / "non_uv_moon_value_diff.csv", index=False
)


issues = []

if integrity["row_count"] != enrich_report.get("number_of_rows", integrity["row_count"]):
    issues.append("Row count mismatch vs enrichment report.")

if integrity.get("ID_plus_presentation_date_unique") is False:
    issues.append("Duplicate ID + presentation_date pairs found.")

if integrity.get("presentation_date_missing", 0) > 0:
    issues.append("Missing or invalid presentation_date detected.")

if len(value_diff_rows) > 0:
    issues.append("Non-UV/moon columns differ from pre-enriched dataset.")

uv_missing = missing_table[missing_table["variable"].isin(uv_cols)]
uv_missing_pct = uv_missing["missing_pct"].max()
if pd.notna(uv_missing_pct) and uv_missing_pct > 2:
    issues.append("UV missingness exceeds 2% in at least one UV column.")

moon_missing_pct = missing_table[missing_table["variable"].isin(moon_cols)]["missing_pct"].max()
if pd.notna(moon_missing_pct) and moon_missing_pct > 0:
    issues.append("Moon feature missingness detected.")


if high_corr_rows:
    issues.append("High collinearity detected (|r| >= 0.9). See high_correlation_pairs.csv.")

summary_lines = []
summary_lines.append("# EDA Summary (Case-only)\n")
summary_lines.append("## Integrity")
summary_lines.append(f"- Rows: {integrity['row_count']}; Columns: {integrity['column_count']}")
summary_lines.append(f"- ID unique: {integrity['ID_unique']}; ID+presentation_date unique: {integrity['ID_plus_presentation_date_unique']}")
summary_lines.append(f"- presentation_date missing: {integrity['presentation_date_missing']}")
summary_lines.append(f"- New columns vs pre-enriched: {len(new_cols)}; Dropped columns: {len(dropped_cols)}")
summary_lines.append("\n## Missingness")
summary_lines.append(f"- UV columns: {len(uv_cols)}; Moon columns: {len(moon_cols)}; Weather columns: {len(weather_cols)}; Clinical columns: {len(clinical_cols)}")
summary_lines.append(f"- UV max missing %: {uv_missing_pct:.2f}" if pd.notna(uv_missing_pct) else "- UV missing %: n/a")
summary_lines.append(f"- Moon max missing %: {moon_missing_pct:.2f}" if pd.notna(moon_missing_pct) else "- Moon missing %: n/a")
uv_missing_cols = uv_missing[uv_missing["missing_pct"] > 0]["variable"].tolist()
if uv_missing_cols:
    summary_lines.append(f"- UV columns with missingness: {', '.join(uv_missing_cols)}")

full_missing_cols = missing_table[missing_table["missing_pct"] == 100]["variable"].tolist()
if full_missing_cols:
    summary_lines.append(f"- Columns with 100% missing: {', '.join(full_missing_cols)}")
summary_lines.append("\n## Seasonality")
if "uv_index_max__day1" in _df.columns and "shortwave_radiation_sum__day1" in _df.columns:
    uv_month_corr = _df[["uv_index_max__day1", "month"]].corr(method="spearman").iloc[0,1]
    sw_month_corr = _df[["shortwave_radiation_sum__day1", "month"]].corr(method="spearman").iloc[0,1]
    summary_lines.append(f"- Spearman corr(month, UV day1): {uv_month_corr:.3f}")
    summary_lines.append(f"- Spearman corr(month, shortwave day1): {sw_month_corr:.3f}")

summary_lines.append("\n## Correlation")
summary_lines.append(f"- High-correlation pairs (|r|>=0.9): {len(high_corr_rows)}")

summary_lines.append("\n## Optional checks")
if optional_notes:
    summary_lines.extend([f"- {n}" for n in optional_notes])
else:
    summary_lines.append("- Gap-fill sensitivity attempted; see uv_gapfill_sensitivity.csv if present.")

summary_lines.append("\n## Potential Issues / Risks")
if issues:
    summary_lines.extend([f"- {i}" for i in issues])
else:
    summary_lines.append("- None detected by automated checks.")

summary_lines.append("\n## Figure Sets")
summary_lines.append("**Thesis-ready**")
summary_lines.append("- uv_histograms.png")
summary_lines.append("- uv_day1_by_month_boxplot.png")
summary_lines.append("- shortwave_by_month_boxplot.png")
summary_lines.append("- uv_vs_shortwave_scatter.png")
summary_lines.append("- uv_weather_correlation_heatmap.png")
summary_lines.append("\n**Internal diagnostics**")
summary_lines.append("- missingness_heatmap.png")
summary_lines.append("- weather_correlation_heatmap.png")
summary_lines.append("- missingness_top30_bar.png")
summary_lines.append("- uv_day1_monthly_z_hist.png")

(Path(OUT_DIR) / "eda_summary.md").write_text("\n".join(summary_lines))

print("EDA complete. Outputs written to", OUT_DIR)
